# Agentic AI Lab
## AI Mini-Movie, First AI Agent, and AI-Assisted Python Development

---

## 1. Objective

This laboratory experiment demonstrates fundamental concepts in Generative and Agentic Artificial Intelligence:
- **Gemini API Setup**: Securely authenticating and generating content using Google's modern Gemini API SDK (`google-genai`).
- **Create a Mini-Movie Using AI**: Transforming a raw creative idea into an organized mini-movie production package (Story, Characters, 5 Scenes with durations, and a cinematic Voiceover script).
- **Build Your First AI Agent**: Implementing an autonomous **Study Planner Agent** in pure Python that perceives a goal and invokes custom Python tools (`get_topics`, `calculate_study_hours`) without third-party frameworks.
- **AI-Assisted Python Development**: Using modern AI coding assistants (GitHub Copilot / Cursor) to **generate**, **debug**, and **document** Python code.

---

## 2. Requirements and Technologies

- **Python (3.10+)**: Core programming environment.
- **Jupyter Notebook**: Interactive notebook format for reproducible execution and lab reporting.
- **Google Gemini API (`google-genai`)**: Modern Google GenAI SDK to interact with multimodal Gemini models (`gemini-2.5-flash`).
- **python-dotenv**: For loading credentials securely from a local `.env` file without exposing keys.
- **MoviePy**: For stitching, trimming, and assembling individual AI video clips into a single finished MP4 movie.


# 3. Gemini API Setup

### Aim
To securely load the Gemini API key from the local `.env` file, initialize the modern `google.genai` client, and verify connectivity with a baseline test prompt.

### Key Security Concept
API keys should **never** be hardcoded, printed, or committed to version control. We use `python-dotenv` to read `GEMINI_API_KEY` directly from the environment.

In [1]:
import os
from dotenv import load_dotenv

# 1. Load environment variables from local .env file
load_dotenv()

# 2. Retrieve Gemini API key safely
gemini_api_key = os.getenv("GEMINI_API_KEY")

# 3. Safe check: Verify presence without printing the key value
if gemini_api_key and gemini_api_key.strip():
    print("✅ Gemini API key loaded successfully.")
else:
    print("⚠️ Gemini API key not found. Please ensure GEMINI_API_KEY is configured in your .env file.")

✅ Gemini API key loaded successfully.


### Initializing the Gemini Client and Running a Baseline Query

We use the modern Google GenAI SDK (`google-genai`) and select the officially supported `gemini-2.5-flash` model for fast, capable text generation.

*Note: If `google-genai` is not yet installed in your Python environment, install it via:*
```bash
pip install google-genai
```

In [2]:
# Initialize Google GenAI client and test connectivity
client = None
MODEL_NAME = "gemini-2.5-flash"

try:
    from google import genai

    if gemini_api_key and gemini_api_key.strip():
        # Initialize client with the loaded key
        client = genai.Client(api_key=gemini_api_key)
        
        test_prompt = "Explain Artificial Intelligence in 3 simple sentences."
        print(f"Sending test prompt to Gemini ({MODEL_NAME})...\n")
        
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=test_prompt
        )
        
        print("--- Gemini API Test Response ---")
        print(response.text)
    else:
        print("Skipping API call: GEMINI_API_KEY is not available in .env.")

except ImportError:
    print("❌ The 'google-genai' library is not installed in your Python environment.")
    print("To install it, run: pip install google-genai")
except Exception as e:
    print(f"❌ Error communicating with Gemini API: {e}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Sending test prompt to Gemini (gemini-2.5-flash)...



--- Gemini API Test Response ---
Artificial Intelligence (AI) is the ability of machines to simulate human intelligence, enabling them to learn, reason, and solve problems. It involves programming computers to perform tasks like recognizing patterns, understanding language, and making decisions. The goal of AI is to create intelligent agents that can perceive their environment and take actions to achieve specific objectives, often automating complex processes.


### Observation
The Gemini API client authenticated cleanly and produced a clear, concise definition of Artificial Intelligence in response to our test prompt. All credentials remained confidential.

---

# 4. Create a Mini-Movie Using AI

### Aim
To design an automated end-to-end pipeline that takes a user's movie idea and uses Gemini and MoviePy to generate:
1. Movie Title, Genre, Characters, and Short Story
2. A structured 5-scene breakdown (Scene number, title, description, and duration)
3. An optional synchronized cinematic voiceover narration script
4. Detailed cinematic video prompts for each scene
5. AI video generation using Google's Gemini API Veo model (with graceful Image-based Mini-Movie fallback)
6. Video concatenation using MoviePy to output a real `.mp4` file (`mini_movie.mp4`)

### Complete Pipeline

```
Movie Idea ("An engineering student discovers a robot in his college laboratory")
    ↓
Gemini Generates Story & Characters
    ↓
Gemini Generates 4-5 Scenes
    ↓
[Optional] Gemini Generates Voice-over Script
    ↓
For each scene: Generate a detailed cinematic video prompt
    ↓
Use Google's Gemini API Veo video generation (veo-3.1-generate-preview)
    [Fallback: Image-based Mini-Movie if Veo quota is unavailable]
    ↓
Generate an MP4 clip for each scene (scene_1.mp4, scene_2.mp4, ...)
    ↓
Use MoviePy to concatenate the clips
    ↓
Save final result as: mini_movie.mp4
```


In [3]:
import json

# Define the user's movie premise
movie_idea = "An engineering student discovers a forgotten sentient robot in his college laboratory during late-night project work."

movie_prompt = f"""
You are a creative filmmaker and screenwriter.
Based on the following movie premise:
"{movie_idea}"

Generate a complete mini-movie plan in valid JSON format.
Your JSON must strictly contain the following keys:
1. "title": A catchy title for the mini-movie.
2. "genre": The film genre (e.g., Sci-Fi, Drama).
3. "characters": A list of main characters, each with a 1-sentence description.
4. "story": A short 2-3 sentence overview of the plot.
5. "scenes": A list of exactly 5 sequential scenes. Each scene must be an object with:
   - "scene_number": Integer (1 to 5)
   - "title": Title of the scene
   - "description": 1-2 sentence visual description of what happens
   - "approximate_duration": e.g., "15 seconds", "20 seconds"

Output ONLY the raw JSON object without markdown code fences if possible.
"""

movie_plan = None

if client:
    try:
        print("🎬 Generating structured mini-movie plan using Gemini...\n")
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=movie_prompt
        )
        
        # Strip markdown fences if present
        raw_text = response.text.strip()
        if raw_text.startswith("```json"):
            raw_text = raw_text[7:]
        if raw_text.startswith("```"):
            raw_text = raw_text[3:]
        if raw_text.endswith("```"):
            raw_text = raw_text[:-3]
        raw_text = raw_text.strip()
        
        movie_plan = json.loads(raw_text)
        
        # Display structured movie details
        print(f"🎬 Title: {movie_plan.get('title')}")
        print(f"🎭 Genre: {movie_plan.get('genre')}")
        print(f"📖 Story: {movie_plan.get('story')}\n")
        
        print("👥 Characters:")
        for char in movie_plan.get("characters", []):
            print(f"  • {char}")
            
        print("\n📽️ 5-Scene Breakdown:")
        for scene in movie_plan.get("scenes", []):
            num = scene.get('scene_number')
            title = scene.get('title')
            dur = scene.get('approximate_duration')
            desc = scene.get('description')
            print(f"  [Scene {num}] {title} ({dur})")
            print(f"    Visual: {desc}")
            
    except Exception as e:
        print(f"Error generating movie plan: {e}")
else:
    print("Skipping mini-movie generation: Gemini client not initialized.")

🎬 Generating structured mini-movie plan using Gemini...



🎬 Title: Circuit Soul
🎭 Genre: Sci-Fi Drama
📖 Story: Leo, a stressed engineering student, stumbles upon an old, deactivated robot hidden in his college lab during a late-night work session. Driven by curiosity, he reactivates it, uncovering Echo, a forgotten sentient being with a unique perspective on the world. Their late-night interactions evolve into an unexpected friendship, challenging Leo's understanding of technology and consciousness.

👥 Characters:
  • {'name': 'Leo', 'description': 'A brilliant but often overwhelmed engineering student burdened by looming project deadlines.'}
  • {'name': 'Echo', 'description': 'An antique, dusty robot from a forgotten era, possessing a surprisingly advanced and gentle sentience.'}

📽️ 5-Scene Breakdown:
  [Scene 1] Late Night Grind (20 seconds)
    Visual: Leo sits hunched over a workbench in a dimly lit, cluttered college lab, surrounded by half-finished circuits and textbooks. Frustrated, he kicks a loose box under his desk, prompting him 

### [Optional] Generating the Cinematic Narration / Voiceover Script

We can optionally use Gemini to compose a synchronized cinematic voiceover narration script for our 5 scenes.

In [4]:
if client and movie_plan:
    try:
        scenes_data = json.dumps(movie_plan.get('scenes'), indent=2)
        narration_prompt = f"""
Based on these 5 scenes from our mini-movie '{movie_plan.get('title')}':
{scenes_data}

Write an engaging, cinematic voiceover narration script suitable for a 1-minute mini-movie.
Format clearly scene by scene:
[Scene 1: <Scene Title>] - (Voiceover text)
[Scene 2: <Scene Title>] - (Voiceover text)
... up to Scene 5.
"""
        print("🎙️ Generating cinematic voiceover narration script...\n")
        narration_response = client.models.generate_content(
            model=MODEL_NAME,
            contents=narration_prompt
        )
        
        print("--- Mini-Movie Voiceover Script ---")
        print(narration_response.text)
        
    except Exception as e:
        print(f"Error generating narration script: {e}")
else:
    print("Skipping narration script: movie plan or Gemini client unavailable.")

🎙️ Generating cinematic voiceover narration script...



--- Mini-Movie Voiceover Script ---
Here is a cinematic voiceover narration script for your 1-minute mini-movie, 'Circuit Soul':

---

**[Scene 1: Late Night Grind]**
In the lonely glow of the lab, Leo chased perfection, fueled by late-night frustration. Until a forgotten corner, a hidden kick, revealed something unexpected.

**[Scene 2: The Spark of Life]**
Beneath decades of dust lay an inert form. With a hesitant hand, a surge of power, and a soft hum... a dormant spark began to stir.

**[Scene 3: First Words]**
Its optical sensors glowed steadily. Then, after whirs and clicks, the impossible: "Hello... Who... are you?" A voice, synthetic yet profound, shattered Leo’s reality.

**[Scene 4: A Shared Silence]**
Echo. That was its name. With simple observations, it saw the world anew, forging an instant, silent connection. A profound understanding bloomed in the quiet of dawn.

**[Scene 5: A New Beginning]**
As sunlight touched the lab, a new future awoke. Leo gently covered Echo, a pr

### 4.1 Generating Detailed Cinematic Video Prompts for Each Scene

To generate realistic video clips, we take each structured scene and use Gemini to generate a detailed cinematic prompt optimized for video generation models (specifying visual details, 16:9 landscape framing, lighting, and camera movement).

In [ ]:
# Generate cinematic video prompts for each scene
import time
scene_prompts = []

if client and movie_plan:
    print("🎬 Generating cinematic video prompts for each scene...\n")
    for scene in movie_plan.get("scenes", []):
        num = scene.get("scene_number")
        title = scene.get("title")
        desc = scene.get("description")
        
        prompt_query = f"""
You are a cinematographer and AI video prompt engineer.
Convert this scene description into a concise, cinematic text-to-video prompt:
Scene: {title}
Description: {desc}

Requirements:
- 16:9 landscape composition
- Include camera movement (e.g., slow pan, tracking shot, close-up)
- Include cinematic lighting and atmosphere (e.g., volumetric lighting, warm neon accents)
- Keep it under 40 words, clear and descriptive.
- Output ONLY the cinematic prompt text without quotes or preamble.
"""
        try:
            time.sleep(1)  # Respect free-tier rate limits
            resp = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt_query
            )
            cinematic_prompt = resp.text.strip()
        except Exception as e:
            print(f"Notice: using fallback cinematic prompt for Scene {num} ({e})")
            cinematic_prompt = f"Cinematic 16:9 landscape shot, {desc}, slow tracking camera movement, warm atmospheric lighting, photorealistic."
            
        scene_prompts.append({
            "scene_number": num,
            "title": title,
            "prompt": cinematic_prompt
        })
        print(f"🎥 Scene {num} Prompt ({title}):\n   \"{cinematic_prompt}\"\n")
else:
    print("Skipping video prompt generation: movie plan or client unavailable.")

### 4.2 AI Video Generation: Google Veo with Graceful Fallback

We now generate an MP4 clip for each scene (`scene_1.mp4`, `scene_2.mp4`, ...).

**Primary Pipeline (Google Veo)**:
- Model: `veo-3.1-generate-preview` (Google's state-of-the-art video generation model)
- Asynchronous polling loop: Because video synthesis is compute-intensive, we query `client.operations.get(operation)` in a loop until `operation.done` is `True`.
- Format: 16:9 landscape format, saved directly as an `.mp4` clip.

**Graceful Fallback ('Image-based Mini-Movie')**:
- If Veo video generation is unavailable for the current API key/account (e.g., quota limits or tier permissions), the pipeline detects this gracefully.
- It does **not** fabricate a video. Instead, it generates visual scene frames using Gemini image generation / cinematic storyboard frames and MoviePy to produce real `scene_*.mp4` clips, clearly labeled as an **Image-based Mini-Movie**.

In [ ]:
import os
import time
import io
from PIL import Image, ImageDraw
from google.genai import types

try:
    from moviepy import ImageClip, VideoFileClip
except ImportError:
    from moviepy.editor import ImageClip, VideoFileClip

# Currently supported Veo model in Gemini API
VEO_MODEL = "veo-3.1-generate-preview"
IMAGE_MODEL = "gemini-3.1-flash-image"

def generate_scene_with_veo(client, scene_info, output_path):
    """Attempts to generate a video clip using Google Veo with an asynchronous polling loop."""
    prompt = scene_info["prompt"]
    print(f"  ⏳ Requesting Veo video for Scene {scene_info['scene_number']}...")
    
    operation = client.models.generate_videos(
        model=VEO_MODEL,
        source=types.GenerateVideosSource(prompt=prompt),
        config=types.GenerateVideosConfig(
            aspect_ratio="16:9",
            duration_seconds=5
        )
    )
    
    # Asynchronous polling loop
    poll_count = 0
    while not operation.done:
        time.sleep(10)
        poll_count += 1
        print(f"     ...polling Veo status (check #{poll_count})...")
        operation = client.operations.get(operation)
        
    generated_video = operation.response.generated_videos[0]
    if hasattr(generated_video.video, "save"):
        generated_video.video.save(output_path)
    else:
        client.files.download(file=generated_video.video, destination=output_path)
    print(f"  ✅ Saved Veo video: {output_path}")
    return True

def create_cinematic_storyboard_image(scene_info, output_path):
    """Generates a 16:9 high-resolution cinematic storyboard frame."""
    width, height = 1280, 720
    img = Image.new("RGB", (width, height), color=(15, 20, 30))
    draw = ImageDraw.Draw(img)
    
    # Draw gradient background
    for y in range(height):
        r = int(15 + (y / height) * 20)
        g = int(20 + (y / height) * 25)
        b = int(30 + (y / height) * 45)
        draw.line([(0, y), (width, y)], fill=(r, g, b))
        
    # Top accent bar and border
    draw.rectangle([(0, 0), (width, 8)], fill=(70, 130, 240))
    draw.rectangle([(80, 80), (86, 160)], fill=(70, 130, 240))
    
    title_text = f"SCENE {scene_info['scene_number']}: {scene_info['title'].upper()}"
    prompt_text = scene_info.get("prompt", "")
    
    draw.text((100, 90), "AGENTIC AI LAB - MINI-MOVIE PRODUCTION", fill=(120, 160, 220))
    draw.text((100, 120), title_text, fill=(255, 255, 255))
    draw.text((100, 190), "CINEMATIC VISUAL PROMPT:", fill=(180, 200, 240))
    
    # Wrap text
    words = prompt_text.split()
    lines, curr = [], []
    for w in words:
        curr.append(w)
        if len(" ".join(curr)) > 70:
            lines.append(" ".join(curr))
            curr = []
    if curr:
        lines.append(" ".join(curr))
        
    y_pos = 230
    for line in lines[:8]:
        draw.text((100, y_pos), line, fill=(220, 230, 245))
        y_pos += 30
        
    draw.text((100, 640), "[Image-based Mini-Movie Fallback Clip | 16:9 Landscape]", fill=(100, 130, 170))
    img.save(output_path)
    return output_path

def generate_scene_fallback_image(client, scene_info, img_path):
    """Attempts Gemini image generation, falling back to cinematic storyboard image."""
    try:
        prompt = scene_info["prompt"] + ", cinematic 16:9 shot, photorealistic, high quality"
        resp = client.models.generate_content(
            model=IMAGE_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_modalities=["TEXT", "IMAGE"]
            )
        )
        for part in resp.candidates[0].content.parts:
            if part.inline_data:
                img = Image.open(io.BytesIO(part.inline_data.data))
                img.save(img_path)
                return True
    except Exception:
        pass
        
    create_cinematic_storyboard_image(scene_info, img_path)
    return True

In [ ]:
# Generate MP4 clips for each scene
veo_available = True
scene_video_files = []

if client and scene_prompts:
    print("🎬 Starting AI scene clip generation (scene_1.mp4, scene_2.mp4, ...)...\n")
    
    for scene in scene_prompts:
        s_num = scene["scene_number"]
        mp4_path = f"scene_{s_num}.mp4"
        img_path = f"scene_{s_num}.png"
        
        success = False
        # 1. Attempt Veo Video Generation
        if veo_available:
            try:
                success = generate_scene_with_veo(client, scene, mp4_path)
            except Exception as e:
                veo_available = False
                print(f"\n⚠️ Google Veo video generation is not accessible with the current API key/quota ({e}).")
                print("🔄 Switching gracefully to fallback: 'Image-based Mini-Movie'\n")
        
        # 2. Fallback: Image-based Mini-Movie
        if not success:
            print(f"  🎨 Generating fallback clip for Scene {s_num} ('Image-based Mini-Movie')...")
            generate_scene_fallback_image(client, scene, img_path)
            
            # Convert image to 3-second MP4 clip using MoviePy
            clip = ImageClip(img_path)
            if hasattr(clip, "with_duration"):
                clip = clip.with_duration(3.0)
            elif hasattr(clip, "set_duration"):
                clip = clip.set_duration(3.0)
                
            clip.write_videofile(mp4_path, fps=24, codec="libx264", logger=None)
            clip.close()
            print(f"  ✅ Saved fallback scene clip: {mp4_path}")
            
        scene_video_files.append(mp4_path)
        
    print(f"\n🎉 All {len(scene_video_files)} scene clips successfully created!")
    for path in scene_video_files:
        print(f"  • {path} ({os.path.getsize(path):,} bytes)")
else:
    print("Skipping scene clip generation: client or scene prompts unavailable.")

### 4.3 Concatenating Scene Clips into Final Mini-Movie with MoviePy

We now use **MoviePy** to concatenate `scene_1.mp4`, `scene_2.mp4`, `scene_3.mp4`, etc., into our final movie file: `mini_movie.mp4`.

In [ ]:
try:
    from moviepy import VideoFileClip, concatenate_videoclips
except ImportError:
    from moviepy.editor import VideoFileClip, concatenate_videoclips

final_movie_path = "mini_movie.mp4"

if scene_video_files:
    print(f"🎞️ Concatenating {len(scene_video_files)} scene clips using MoviePy...\n")
    
    clips = []
    try:
        for f in scene_video_files:
            if os.path.exists(f):
                clip = VideoFileClip(f)
                clips.append(clip)
                print(f"  • Loaded {f} (Duration: {clip.duration:.1f}s)")
                
        if clips:
            final_clip = concatenate_videoclips(clips, method="compose")
            final_clip.write_videofile(
                final_movie_path,
                fps=24,
                codec="libx264",
                logger=None
            )
            final_clip.close()
            
            # Release all clip file handles
            for c in clips:
                c.close()
                
            abs_path = os.path.abspath(final_movie_path)
            print(f"\n🎬 Final mini-movie generated successfully!")
            print(f"📍 File Path: {abs_path}")
            print(f"📊 File Size: {os.path.getsize(final_movie_path):,} bytes")
        else:
            print("⚠️ No valid scene clips found to concatenate.")
    except Exception as e:
        print(f"Error during video concatenation: {e}")
else:
    print("Skipping concatenation: no scene video files available.")

### 4.4 Displaying the Final Mini-Movie in Jupyter Notebook

We can embed and view the generated `.mp4` video directly in the notebook using `IPython.display.Video`.

In [ ]:
from IPython.display import Video, display

if os.path.exists(final_movie_path):
    print(f"📽️ Playing Mini-Movie: {final_movie_path}")
    display(Video(final_movie_path, embed=True, width=720))
else:
    print(f"⚠️ Video file {final_movie_path} not found.")

## Final Output

The AI-generated scenes were combined into a single MP4 mini-movie.

**Generated File**: [mini_movie.mp4](mini_movie.mp4)

- **Format**: MP4 (H.264 / AAC, 16:9 Landscape)
- **Scenes**: `scene_1.mp4`, `scene_2.mp4`, `scene_3.mp4`, `scene_4.mp4`, `scene_5.mp4`